<a href="https://colab.research.google.com/github/Skyler-bot-code/Bivariate-and-Multivariate-Linear-Regression-Analysis-and-Model-Depiction-/blob/main/Gradient_Descent_Algorithm_of_Bivariate_and_Multivariate_Linear_Regression_in_Heart_Disease_Correlation_Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [242]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shyamnadhs/heart-disease-prediction-dataset")

print("Path to dataset files:", path)


# Data imported from Kaggle by Shyam Nadh S

Using Colab cache for faster access to the 'heart-disease-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/heart-disease-prediction-dataset


In [243]:
import os

# List the contents of the downloaded dataset directory
print(os.listdir(path))

['disease_prediction.csv']


In [244]:
import pandas as pd

# Changeg the raw data for different analysis
# The FileNotFoundError indicates the previous filename guess was incorrect.
# Reverting to the most common pattern for this Kaggle dataset:
# the CSV file name matching the directory name, retaining hyphens.
url_raw = path + "/disease_prediction.csv" # Corrected filename by adding a '/' for proper path concatenation

df1 = pd.read_csv(url_raw, header=None)

df1

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,patient_id,age,gender,glucose_mg_dl,cholesterol_mg_dl,systolic_bp,diastolic_bp,bmi,heart_rate,smoking,alcohol_consumption,physical_activity,family_history,disease
1,1,32,Male,101,235,152,79,28.5,73,No,Yes,Low,Yes,Yes
2,2,31,Male,124,191,134,77,33.9,71,No,Yes,Low,Yes,Yes
3,3,45,Male,57,141,114,71,27.2,79,Yes,Yes,Low,No,No
4,4,75,Female,69,268,120,82,21.5,61,Yes,Yes,Medium,No,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,996,24,Male,105,237,86,86,30.9,69,Yes,Yes,Medium,No,No
997,997,40,Male,120,219,133,72,34.3,92,Yes,Yes,Medium,No,No
998,998,44,Female,114,273,114,74,36.0,76,No,No,Low,No,Yes
999,999,31,Male,95,231,130,72,28.2,81,Yes,Yes,Medium,Yes,No


In [245]:
import pandas as pd

# Set the first row as the column headers by converting it to a list of strings and stripping whitespace
# This ensures that column names are clean and correctly recognized.
df1.columns = [str(col).strip() for col in df1.iloc[0].tolist()]

# Drop the first row from the DataFrame as it's now the header, and reset the index
df1 = df1[1:].reset_index(drop=True)

# bmi is dependent variable
columns_to_extract = ['heart_rate','bmi','age']
df = df1[columns_to_extract].copy()

df.head()

,heart_rate,bmi,age
0,73,28.5,32
1,71,33.9,31
2,79,27.2,45
3,61,21.5,75
4,73,23.3,53


In [246]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

print("Loaded Data Head:")
display(df.head())

# Select target variable (last column)
Y_raw = df.iloc[:, -1].values.reshape(-1, 1)
# Convert Y_raw to numeric type (float)
Y_raw = Y_raw.astype(float)

# Select feature variables (all columns except the last one)
X_features = df.iloc[:, :-1].values

print("\nShape of X_features (before scaling):", X_features.shape)
print("First 5 rows of X_features (before scaling):\n", X_features[:5])
print("\nShape of Y_raw:", Y_raw.shape)
print("First 5 rows of Y_raw:\n", Y_raw[:5])


# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler to your features and transform them
X_scaled_features = scaler.fit_transform(X_features)

# Add the intercept column back to the scaled features, ensuring X_scaled is ready for the gradient descent algorithm (which expects an intercept term)
X_scaled = np.hstack((np.ones((X_scaled_features.shape[0], 1)), X_scaled_features))

print("\nShape of X_scaled (with intercept column):", X_scaled.shape)
print("First 10 rows of X_scaled:\n", X_scaled[:10])

Loaded Data Head:


,heart_rate,bmi,age
0,73,28.5,32
1,71,33.9,31
2,79,27.2,45
3,61,21.5,75
4,73,23.3,53



Shape of X_features (before scaling): (1000, 2)
First 5 rows of X_features (before scaling):
 [['73' '28.5']
 ['71' '33.9']
 ['79' '27.2']
 ['61' '21.5']
 ['73' '23.3']]

Shape of Y_raw: (1000, 1)
First 5 rows of Y_raw:
 [[32.]
 [31.]
 [45.]
 [75.]
 [53.]]

Shape of X_scaled (with intercept column): (1000, 3)
First 10 rows of X_scaled:
 [[ 1.         -0.29985768  0.50610791]
 [ 1.         -0.50191811  1.64153903]
 [ 1.          0.30632362  0.23276338]
 [ 1.         -1.51222028 -0.96574725]
 [ 1.         -0.29985768 -0.58727021]
 [ 1.         -1.31015984 -0.65034971]
 [ 1.          0.10426318  0.19071038]
 [ 1.          1.01353513 -0.14571366]
 [ 1.          0.60941426  0.33789589]
 [ 1.          0.91250491  1.07382347]]


## Applying Grdient Descent Algorithm to minimize the Cost Function ##

In [247]:
# @title Enter the number of epochs (number of iteration)
epochs = 88 # @param {"type":"slider","min":0,"max":100,"step":1}
import numpy as np

# Define X and Y for gradient descent using the prepared scaled data
X_gd = X_scaled
Y_gd = Y_raw
m = X_gd.shape[0] # Number of data points
n_parameters = X_gd.shape[1] # Number of parameters (intercept + features)

# w to save the current intercept and slope (now generalizes to all parameters)
# Initialize w as a column vector (n_parameters, 1) to ensure consistent dimensions
w = np.zeros((n_parameters, 1))

# saving for plot
# list of cost from each epoch (iteration)
cost_list = []

# Save the coefficients at each epoch
w_history = []

########## to be controlled by the user #########
# No longer setting individual w[0], w[1] as the number of parameters can vary
# w = np.zeros(n_parameters) # Already initialized above

epochs = epochs # number of epochs (iteration)
lr = 0.2 # learning rate, how fast do you want to descent
######################################################

# save the first list of ws
w_history.append(w.copy().flatten()) # Store as 1D array for consistency with previous plotting if any

print(f"Gradient Descent for {epochs} epochs on new data")
print("Epoch           Parameters (w)              Cost")

# performing gradient descent numerically
for i in range(epochs):
  # @ for matrix multiplication
  # compute Xw = predicted values

  # Linear Regression Mathematical Relationship
  Y_pred = X_gd @ w # X_gd (m, n) @ w (n, 1) -> Y_pred (m, 1)

  # compute the cost (Mean Squared Error scaled by 1/2), Sum of Squared Errors (SSE) calculations
  cost_val = (1/(2 * m)) * np.sum((Y_pred - Y_gd)**2)
  # save this in the cost list
  cost_list.append(cost_val)

  # Print current state
  print(f"{i: 4.0f} {w.flatten()} {cost_val: 13.5f}") # Print flattened w for readability

  # make the machine learn (update the ws with learning rate lr)
  # X_gd.T (n, m) @ (Y_pred - Y_gd) (m, 1) -> gradient (n, 1)
  w = w - lr * (1/m) * (X_gd.T @ (Y_pred - Y_gd)) # Removed .flatten()

  # save the w to the list
  w_history.append(w.copy().flatten()) # Store as 1D array

print(f"{epochs: 4.0f} {w.flatten()} {cost_list[-1]: 13.5f}")

Gradient Descent for 88 epochs on new data
Epoch           Parameters (w)              Cost
   0 [0. 0. 0.]    1506.97300
   1 [10.238       0.11349896  0.0588208 ]    1035.22457
   2 [18.4284      0.20429416  0.10586978]     733.30557
   3 [24.98072     0.27692715  0.14350285]     540.07742
   4 [30.222576   0.335031   0.1736044]     416.41141
   5 [34.4160608   0.38151205  0.19768172]     337.26516
   6 [37.77084864  0.41869527  0.21694044]     286.61157
   7 [40.45467891  0.44844055  0.23234491]     254.19326
   8 [42.60174313  0.47223573  0.24466648]     233.44555
   9 [44.3193945   0.49127104  0.25452213]     220.16702
  10 [45.6935156   0.50649862  0.26240537]     211.66875
  11 [46.79281248  0.51868016  0.26871093]     206.22987
  12 [47.67224999  0.52842497  0.27375456]     202.74898
  13 [48.37579999  0.53622047  0.2777888 ]     200.52121
  14 [48.93863999  0.5424566   0.28101567]     199.09544
  15 [49.38891199  0.54744529  0.28359675]     198.18294
  16 [49.74912959  0.55143

In [248]:
print('\n      --- Updated Gradient Descent Results (with All Features) ---\n')
print(f"Final Estimated Parameters (Intercept, Slope1, Slope2): {w_history[-1]}")
print(f"Final Cost (Mean Squared Error):   {cost_list[-1]:.5f}")


      --- Updated Gradient Descent Results (with All Features) ---

Final Estimated Parameters (Intercept, Slope1, Slope2): [51.18999985  0.56739568  0.29391264]
Final Cost (Mean Squared Error):   196.56073
